串口读出来的 3 个字节：

```python
raw = (b1 << 16) | (b2 << 8) | b3
```

拼接后的结果：

```text
既不是“源码”
也不是“Python意义上的负数”
```

而是：

```text
ADS1299 输出的 24位二进制补码(bit pattern)
```

更准确地说：

```text
它是“补码的原始位模式”
```

---

# 一、什么叫“源码”“补码”

以 8 位整数为例。

---

## +5

二进制：

```text
00000101
```

源码、反码、补码都一样。

---

## -5

---

### 源码（sign-magnitude）

```text
10000101
```

最高位表示符号。

现代 CPU 基本不用。

---

### 补码（two's complement）

步骤：

```text
+5: 00000101
取反:11111010
+1 :11111011
```

因此：

```text
-5 的补码 = 11111011
```

现代 CPU / ADC / DSP 几乎都用补码。

---

# 二、ADS1299 发出来的是什么

ADS1299 是：

```text
24-bit signed ADC
```

它输出的是：

```text
24位补码
```

例如：

```text
11111111 11111111 11111111
```

表示：

```text
-1
```

---

# 三、但 Python 不知道它是补码

这是关键。

你拼接后：

```python
raw = (b1 << 16) | (b2 << 8) | b3
```

Python 只会认为：

```text
这是普通正整数
```

例如：

```text
0xFFFFFF
=
16777215
```

Python 不会自动认为：

```text
这是24位 signed int
```

因为：

```text
Python int 没有固定24位
```

---

# 四、所以需要“符号扩展”

代码：

```python
if raw & 0x800000:
    raw -= 0x1000000
```

作用：

```text
把“24位补码位模式”
转换成
Python真正的负整数
```

---

# 五、举例最清楚

---

## ADS1299 发来

```text
0xFFFFFF
```

二进制：

```text
11111111 11111111 11111111
```

在：

```text
24位补码系统
```

中：

```text
表示 -1
```

---

## 但 Python 拼接后

得到：

```python
raw = 16777215
```

因为 Python 当它是：

```text
无符号数
```

---

## 然后代码：

```python
raw -= 0x1000000
```

即：

```python
16777215 - 16777216
= -1
```

成功恢复。

---

# 六、本质上发生了什么

串口里的数据：

```text
只是 bit pattern（二进制位）
```

例如：

```text
11111111 11111111 11111111
```

它本身：

```text
没有“正负”
```

正负取决于：

```text
你如何解释这些位
```

---

# 七、当前代码的解释过程

---

## 第一步：拼接

```python
raw = ...
```

得到：

```text
24位bit pattern
```

---

## 第二步：检查最高位

```python
raw & 0x800000
```

判断：

```text
符号位是否为1
```

---

## 第三步：转成 Python signed int

```python
raw -= 2^24
```

---

# 八、所以最终答案

串口读出来并拼接好的：

```text
raw
```

在执行：

```python
if raw & 0x800000:
```

之前：

```text
它是：
“24位补码的位模式”
```

不是：

```text
源码
```

也不是：

```text
Python已经识别好的负数
```

而：

```text
是一串需要你自己解释的24位补码数据
```

---

# 九、可以理解成

```text
ADC:
“这是24位补码”
```

Python：

```text
“我不知道你是24位补码，
我只看到一个正整数”
```

于是你必须：

```text
手动符号扩展
```

把它恢复成真正负数。
